# ORACLE: Data Exploration

Explore raw scRNA-seq data for cancer attractor mapping.
Covers quality control, normalization, dimensionality reduction, and cell state annotation.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor='white')
DATA_DIR = Path('../data')
RESULTS_DIR = Path('../data/processed')

## 1. Load Data

In [ ]:
from oracle.data.loaders import OracleDataLoader

loader = OracleDataLoader(data_dir=str(DATA_DIR))
adata = loader.load_scrnaseq(cancer_type='colorectal', sample_id='GSE132465')
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')
print(adata)

## 2. Quality Control

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0], show=False)
sc.pl.violin(adata, 'total_counts', ax=axes[1], show=False)
sc.pl.violin(adata, 'pct_counts_mt', ax=axes[2], show=False)
plt.tight_layout()
plt.show()

print(f'Cells before QC: {adata.n_obs}')
adata = adata[adata.obs['n_genes_by_counts'] > 200]
adata = adata[adata.obs['pct_counts_mt'] < 20]
print(f'Cells after QC: {adata.n_obs}')

## 3. Normalization & HVG Selection

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=3000)
print(f'Highly variable genes: {adata.var.highly_variable.sum()}')
sc.pl.highly_variable_genes(adata)

## 4. Dimensionality Reduction

In [ ]:
adata_hvg = adata[:, adata.var.highly_variable]
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata_hvg, n_pcs=50, n_neighbors=15)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg, resolution=0.5)

sc.pl.umap(adata_hvg, color=['leiden'], title='Leiden clusters')

## 5. Cell State Annotation

In [ ]:
from oracle.cam.preprocessing import CancerAttractionPreprocessor, CAMConfig

config = CAMConfig(cancer_type='colorectal', tissue='colon')
preprocessor = CancerAttractionPreprocessor(config)
adata_processed = preprocessor.run(adata)

sc.pl.umap(
    adata_processed,
    color=['cell_state', 'leiden'],
    palette={'normal': '#2196F3', 'cancer': '#F44336', 'transitional': '#FF9800'},
    title=['Cell State', 'Leiden Clusters']
)

print('Cell state distribution:')
print(adata_processed.obs['cell_state'].value_counts())

## 6. Save Processed Data

In [ ]:
out_path = RESULTS_DIR / 'anndata' / 'colorectal_GSE132465_processed.h5ad'
out_path.parent.mkdir(parents=True, exist_ok=True)
adata_processed.write_h5ad(out_path)
print(f'Saved to {out_path}')